# Multi-model metric comparison visualizations

Compare metric profiles across four LLMs, all datasets, HG/MG splits, and token prefix lengths. This notebook exports three complementary visualization families: trajectory small multiples, heatmaps, and token-length distribution violins.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from notebooks.utils.logprob_metrics_distributions_utils import (
    default_length_metrics_path,
    load_length_metric_table,
)

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
OUTPUT_ROOT = DATA_ROOT / "plots" / "multi_model_metric_comparison"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
METRICS = ["perplexity", "d_score_top20", "prompt_mu_score", "prompt_sigma_score"]
MODELS = ["gemma_3_1b_it", "gemma_3_270m_it", "littlelamb", "smollm2_360m"]
SPLITS = ["HG", "MG"]
MODEL_PALETTE = dict(zip(MODELS, sns.color_palette("viridis", len(MODELS))))
metric_df = load_length_metric_table(default_length_metrics_path(PROCESSED_ROOT))
metric_df = metric_df.loc[metric_df["model"].isin(MODELS)].copy()
metric_df["model"] = pd.Categorical(metric_df["model"], categories=MODELS, ordered=True)
metric_df["split"] = pd.Categorical(metric_df["split"], categories=SPLITS, ordered=True)
datasets = sorted(metric_df["dataset"].dropna().unique())
DATASET_PALETTE = dict(zip(datasets, sns.color_palette("tab10", len(datasets))))
print(f"Loaded {len(metric_df):,} rows")
print(f"Token lengths: {sorted(metric_df['token_length'].dropna().unique())}")
print(f"Datasets: {', '.join(datasets)}")

## 1. Dataset trajectory small multiples

Each dataset is a colored line. The line is the dataset/model/split median over texts, with an IQR band. Overlapping HG lines indicate a stable cross-dataset profile; separated MG lines indicate dataset-specific behavior.

In [ ]:
trajectory_summary = (
    metric_df.groupby(["dataset", "model", "split", "token_length"], observed=True)[METRICS]
    .agg(["median", lambda values: values.quantile(0.25), lambda values: values.quantile(0.75)])
)
trajectory_summary.columns = ["__".join(column) for column in trajectory_summary.columns]
trajectory_summary = trajectory_summary.reset_index()
trajectory_summary.to_csv(OUTPUT_ROOT / "dataset_spread_summary.csv", index=False)

for metric in METRICS:
    fig, axes = plt.subplots(len(MODELS), len(SPLITS), figsize=(16, 14), sharex=True, squeeze=False)
    for row, model in enumerate(MODELS):
        for col, split in enumerate(SPLITS):
            ax = axes[row, col]
            view = trajectory_summary.loc[
                (trajectory_summary["model"] == model) & (trajectory_summary["split"] == split)
            ]
            for dataset in datasets:
                line = view.loc[view["dataset"] == dataset].sort_values("token_length")
                if line.empty:
                    continue
                median = line[f"{metric}__median"]
                q25 = line[f"{metric}__<lambda_0>"]
                q75 = line[f"{metric}__<lambda_1>"]
                color = DATASET_PALETTE[dataset]
                ax.plot(line["token_length"], median, color=color, linewidth=2, label=dataset)
                ax.fill_between(line["token_length"], q25, q75, color=color, alpha=0.10)
            if metric == "perplexity":
                ax.set_yscale("log")
            ax.set_title(f"{model} | {split}")
            ax.set_xlabel("Token prefix length")
            ax.set_ylabel(metric)
            ax.set_xlim(0, 1200)
            ax.grid(True, alpha=0.3)
    handles = [plt.Line2D([0], [0], color=DATASET_PALETTE[dataset], linewidth=2) for dataset in datasets]
    fig.legend(handles, datasets, title="Dataset", loc="center left", bbox_to_anchor=(0.88, 0.5))
    fig.suptitle(f"{metric}: cross-dataset trajectories by model and split", fontsize=16, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 0.86, 0.97))
    fig.savefig(OUTPUT_ROOT / f"trajectories__{metric}.png", dpi=300, bbox_inches="tight")
    plt.show()

## 2. Dataset-by-token-length heatmaps

Each cell is the median metric value for one dataset and token length. Model rows and HG/MG columns make it easy to compare the full profile without line overlap.

In [ ]:
for metric in METRICS:
    heatmap_summary = (
        metric_df.groupby(["dataset", "model", "split", "token_length"], observed=True)[metric]
        .median()
        .reset_index()
    )
    finite_values = heatmap_summary[metric].dropna()
    norm = mcolors.LogNorm(vmin=finite_values.min(), vmax=finite_values.max()) if metric == "perplexity" else None
    fig, axes = plt.subplots(len(MODELS), len(SPLITS), figsize=(16, 14), squeeze=False)
    for row, model in enumerate(MODELS):
        for col, split in enumerate(SPLITS):
            ax = axes[row, col]
            view = heatmap_summary.loc[(heatmap_summary["model"] == model) & (heatmap_summary["split"] == split)]
            table = view.pivot(index="dataset", columns="token_length", values=metric).reindex(index=datasets)
            sns.heatmap(table, ax=ax, cmap="magma", norm=norm, cbar=row == 0 and col == len(SPLITS) - 1, linewidths=0.4)
            ax.set_title(f"{model} | {split}")
            ax.set_xlabel("Token prefix length")
            ax.set_ylabel("Dataset")
    fig.suptitle(f"{metric}: median profile heatmaps", fontsize=16, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(OUTPUT_ROOT / f"heatmaps__{metric}.png", dpi=300, bbox_inches="tight")
    plt.show()

## 3. Full distributions by token length

Violin plots retain the text-level spread instead of reducing each dataset to one median. Rows are representative token lengths, columns are HG/MG, x is dataset, and color is model. The trajectory and heatmap options retain every available token length.

In [ ]:
available_token_lengths = sorted(metric_df["token_length"].dropna().unique())
distribution_token_lengths = [
    length for length in [50, 300, 600, 900, 1200]
    if length in available_token_lengths
]
for metric in METRICS:
    view = metric_df.dropna(subset=[metric]).copy()
    fig, axes = plt.subplots(len(distribution_token_lengths), len(SPLITS), figsize=(18, max(7, 2.8 * len(distribution_token_lengths))), squeeze=False, sharex=True)
    for row, token_length in enumerate(distribution_token_lengths):
        for col, split in enumerate(SPLITS):
            ax = axes[row, col]
            subset = view.loc[(view["token_length"] == token_length) & (view["split"] == split)]
            sns.violinplot(
                data=subset, x="dataset", y=metric, hue="model", order=datasets, hue_order=MODELS,
                palette=MODEL_PALETTE, inner="quartile", cut=0, scale="width", dodge=True, legend=False, ax=ax,
            )
            if metric == "perplexity":
                ax.set_yscale("log")
            ax.set_title(f"{token_length} tokens | {split}")
            ax.set_xlabel("Dataset")
            ax.set_ylabel(metric)
            ax.tick_params(axis="x", rotation=35)
    handles = [plt.Line2D([0], [0], color=MODEL_PALETTE[model], linewidth=5) for model in MODELS]
    fig.legend(handles, MODELS, title="Model", loc="center left", bbox_to_anchor=(0.88, 0.5))
    fig.suptitle(f"{metric}: text-level distributions by token length", fontsize=16, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 0.86, 0.97))
    fig.savefig(OUTPUT_ROOT / f"distributions_by_token_length__{metric}.png", dpi=300, bbox_inches="tight")
    plt.show()

## 4. Full distributions by token length

LLMs are shown on the x-axis, while dataset colors identify the text distributions within each HG/MG and token-length panel.

In [ ]:
for metric in METRICS:
    view = metric_df.dropna(subset=[metric]).copy()
    fig, axes = plt.subplots(len(distribution_token_lengths), len(SPLITS), figsize=(18, max(7, 2.8 * len(distribution_token_lengths))), squeeze=False, sharex=True)
    for row, token_length in enumerate(distribution_token_lengths):
        for col, split in enumerate(SPLITS):
            ax = axes[row, col]
            subset = view.loc[(view["token_length"] == token_length) & (view["split"] == split)]
            sns.violinplot(
                data=subset, x="model", y=metric, hue="dataset", order=MODELS, hue_order=datasets,
                palette=DATASET_PALETTE, inner="quartile", cut=0, scale="width", dodge=True, legend=False, ax=ax,
            )
            if metric == "perplexity":
                ax.set_yscale("log")
            ax.set_title(f"{token_length} tokens | {split}")
            ax.set_xlabel("LLM")
            ax.set_ylabel(metric)
            ax.tick_params(axis="x", rotation=35)
    handles = [plt.Line2D([0], [0], color=DATASET_PALETTE[dataset], linewidth=5) for dataset in datasets]
    fig.legend(handles, datasets, title="Dataset", loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=len(datasets))
    fig.suptitle(f"{metric}: text-level distributions by token length and LLM", fontsize=16, fontweight="bold")
    fig.tight_layout(rect=(0, 0.05, 1, 0.97))
    fig.savefig(OUTPUT_ROOT / f"distributions_by_token_length__model_x__dataset_hue__{metric}.png", dpi=300, bbox_inches="tight")
    plt.show()